<!--
  Licensed to the Apache Software Foundation (ASF) under one or more
  contributor license agreements.  See the NOTICE file distributed with
  this work for additional information regarding copyright ownership.
  The ASF licenses this file to You under the Apache License, Version 2.0
  (the "License"); you may not use this file except in compliance with
  the License.  You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

  Unless required by applicable law or agreed to in writing, software
  distributed under the License is distributed on an "AS IS" BASIS,
  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
  See the License for the specific language governing permissions and
  limitations under the License.
-->

# Hudi Batch Vector Search — Certification Demo

Certifies the `hudi_vector_search_batch` TVF (RFC-102) at non-trivial scale
against a **numpy ground-truth oracle**.

Sibling to [`02_sql_vector_search.ipynb`](02_sql_vector_search.ipynb), which
runs the **single-query** TVF on one query vector. This notebook runs the
**table-to-table** form: for each row in a query table, find the top-K
nearest rows in a corpus table.

```sql
SELECT *
FROM hudi_vector_search_batch(
    'pets_batch_corpus_<format>',  'embedding',
    'pets_batch_queries_<format>', 'embedding',
    k, 'cosine')
```

Flow:

1. Load `N_CORPUS + N_QUERIES` Oxford-IIIT Pet images.
2. Generate L2-normalized embeddings (`mobilenetv3_small_100`).
3. Split into corpus + held-out queries (disjoint).
4. Write each as its own Hudi table (`VECTOR(1024)`, `BLOB`).
5. Run `hudi_vector_search_batch` — collect to driver.
6. **Oracle**: recompute the cosine distance matrix in numpy from the same
   embeddings; assert TVF's top-K per query matches.
7. Render a panel: one row per query × its top-K matches.

Mirrors [`hudi_vector_search_batch_demo.py`](../hudi_vector_search_batch_demo.py)
cell-for-cell. The `.py` is the source of truth; this notebook is for live
walkthroughs.

## 1. Toggles

| Variable | Effect |
|---|---|
| `BASE_FILE_FORMAT` | `"parquet"` or `"lance"` — flips Hudi's base file format. |
| `N_CORPUS`         | Corpus row count (Hudi table the search reads against). |
| `N_QUERIES`        | Query row count (Hudi table whose rows are searched for). |
| `TOP_K`            | Nearest neighbors returned per query. |
| `EMBEDDING_MODEL`  | `timm` backbone — default `mobilenetv3_small_100` (1024-d). |

A `1000 × 20 × k=5` run produces a 20,000-row cross-join intermediate inside
`buildBatchQueryPlan` — large enough to exercise the broadcast + window-rank
machinery while still completing in under a minute on the default 4 GB driver
heap.

In [ ]:
# ===== EDIT THESE =====
BASE_FILE_FORMAT = "parquet"   # "parquet" or "lance"
N_CORPUS         = 1000
N_QUERIES        = 20
TOP_K            = 5
EMBEDDING_MODEL  = "mobilenetv3_small_100"

assert BASE_FILE_FORMAT in {"parquet", "lance"}
assert N_QUERIES >= 1 and N_CORPUS >= TOP_K

## 2. Pre-flight cleanup

Wipe `/tmp/` paths and `spark-warehouse/` so each run is idempotent. `DROP
TABLE IF EXISTS` only removes the catalog entry; the data dir and `.hoodie/`
timeline at `LOCATION` persist across runs.

In [ ]:
import shutil
from pathlib import Path

for pattern in [
    "/tmp/hudi_batch_*_pets",
    "/tmp/staging_pets_batch_*.parquet",
]:
    for p in Path("/").glob(pattern.lstrip("/")):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
        elif p.is_file():
            p.unlink(missing_ok=True)

shutil.rmtree("spark-warehouse", ignore_errors=True)
print("✓ Wiped /tmp/hudi_batch_*_pets, staging Parquet, spark-warehouse")

## 3. Pre-JVM env (driver heap)

PySpark local-mode bakes the driver heap into the JVM at launch — by the
time `pyspark` is imported it can't be raised via `SparkSession.config(...)`.
Set it here, before any other imports.

In [ ]:
import os

DRIVER_MEMORY = "4g"   # bump to "8g" for N_CORPUS >= 5000

os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    f"--driver-memory {DRIVER_MEMORY} --conf spark.driver.maxResultSize=2g pyspark-shell",
)

## 4. Imports

In [ ]:
import io
import sys
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import timm
from sklearn.preprocessing import normalize
from PIL import Image
import matplotlib.pyplot as plt

from torchvision.datasets import OxfordIIITPet
from pyspark.sql import SparkSession

## 5. Configuration (derived from toggles)

In [ ]:
CONFIG = {
    "dataset": "OxfordIIITPet",
    "base_file_format": BASE_FILE_FORMAT,
    "corpus_table_path": f"/tmp/hudi_batch_corpus_{BASE_FILE_FORMAT}_pets",
    "corpus_table_name": f"pets_batch_corpus_{BASE_FILE_FORMAT}",
    "queries_table_path": f"/tmp/hudi_batch_queries_{BASE_FILE_FORMAT}_pets",
    "queries_table_name": f"pets_batch_queries_{BASE_FILE_FORMAT}",
    "n_corpus": N_CORPUS,
    "n_queries": N_QUERIES,
    "top_k": TOP_K,
    "embedding_model": EMBEDDING_MODEL,
    "output_dir": "./outputs",
    "panel_filename": f"hudi_vector_search_batch_{BASE_FILE_FORMAT}_results.png",
    "oracle_distance_tol": 1e-5,
}

BLOB_REFERENCE_CAST = "struct<external_path:string,offset:bigint,length:bigint,managed:boolean>"

for k, v in CONFIG.items():
    print(f"  {k:22s}: {v}")

## 6. Resolve Hudi (and optionally Lance) jars

Defaults to `~/Downloads/`. Set `HUDI_BUNDLE_JAR` and/or `LANCE_BUNDLE_JAR`
*before launching jupyter* if the jars live elsewhere.

In [ ]:
def default_hudi_bundle_jar() -> str:
    return str(Path.home() / "Downloads" / "hudi-spark3.5-bundle_2.12-1.2.0-rc1.jar")

def default_lance_bundle_jar() -> str:
    return str(Path.home() / "Downloads" / "lance-spark-bundle-3.5_2.12-0.4.0.jar")

def resolve_jars(base_file_format: str) -> str:
    hudi_jar = os.getenv("HUDI_BUNDLE_JAR", default_hudi_bundle_jar())
    if not Path(hudi_jar).is_file():
        sys.exit(f"ERROR: HUDI_BUNDLE_JAR does not exist at {hudi_jar}")
    if base_file_format != "lance":
        return hudi_jar
    lance_jar = os.getenv("LANCE_BUNDLE_JAR", default_lance_bundle_jar())
    if not Path(lance_jar).is_file():
        sys.exit(f"ERROR: LANCE_BUNDLE_JAR does not exist at {lance_jar}")
    return f"{hudi_jar},{lance_jar}"

jars = resolve_jars(CONFIG["base_file_format"])
print(f"✓ Jars: {jars}")

## 7. Spark session

Same four Hudi-mandatory configs as the other demos: Kryo serializer,
`HoodieSparkSessionExtension` (registers `hudi_vector_search_batch` and the
`VECTOR(N)`/`BLOB` type parsers), and `HoodieCatalog`.

In [ ]:
spark = (
    SparkSession.builder
    .appName("Hudi-Vector-Search-Batch-Demo")
    .config("spark.jars", jars)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.sql.session.timeZone", "UTC")
    .config("hoodie.read.blob.inline.mode", "CONTENT")
    .config("spark.default.parallelism", "2")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("✓ Spark session ready")

## 8. Load Oxford-IIIT Pet

Load `N_CORPUS + N_QUERIES` images in one pass. We hold them in Python so
the oracle (cell 16) can recompute the distance matrix from the exact same
embeddings that were written to Hudi.

In [ ]:
print(f"Loading dataset: Oxford-IIIT Pet ({CONFIG['n_corpus'] + CONFIG['n_queries']} samples)...")
root = os.path.expanduser("~/.cache/torchvision")
ds = OxfordIIITPet(root=root, split="trainval", download=True)
class_names = ds.classes

rng = np.random.default_rng()
total = CONFIG['n_corpus'] + CONFIG['n_queries']
indices = rng.choice(len(ds), size=min(total, len(ds)), replace=False)

data = []
for idx in indices:
    img, label = ds[int(idx)]
    img = img.convert("RGB")
    bio = io.BytesIO()
    img.save(bio, format="PNG")
    img_bytes = bio.getvalue()
    w, h = img.size
    category = class_names[label] if isinstance(class_names, list) else str(label)
    data.append({
        "image_id": f"pets_{int(idx):06d}",
        "category": category,
        "category_sanitized": category.replace("/", "_"),
        "label": int(label),
        "description": f"{category} from Oxford-IIIT Pet",
        "image_bytes_raw": img_bytes,
        "width": int(w),
        "height": int(h),
    })
print(f"✓ Loaded {len(data)} images")

## 9. Generate embeddings (timm)

`num_classes=0` strips the classifier head so `model(x)` returns feature
vectors. `sklearn.preprocessing.normalize` L2-normalizes each embedding —
cosine distance is then `1 - dot_product`, which the oracle exploits in
cell 16.

In [ ]:
print(f"Loading embedding model: {CONFIG['embedding_model']}...")
model = timm.create_model(CONFIG["embedding_model"], pretrained=True, num_classes=0)
model.eval()
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

print(f"Generating embeddings for {len(data)} images...")
images = [transform(Image.open(io.BytesIO(item["image_bytes_raw"])).convert("RGB")) for item in data]
batch = torch.stack(images)
with torch.no_grad():
    feats = model(batch).detach().cpu().numpy()
feats = normalize(feats)
for i, item in enumerate(data):
    item["embedding"] = feats[i].tolist()
embedding_dim = int(feats.shape[1])
print(f"✓ Generated embeddings (dimension: {embedding_dim})")

## 10. Split into corpus + queries

First `N_CORPUS` → corpus, last `N_QUERIES` → queries. Order is randomized
by `rng.choice(replace=False)`, so a simple slice gives disjoint subsets.

In [ ]:
corpus = data[:CONFIG["n_corpus"]]
queries = data[CONFIG["n_corpus"]:CONFIG["n_corpus"] + CONFIG["n_queries"]]
assert {r["image_id"] for r in corpus}.isdisjoint({r["image_id"] for r in queries})
print(f"✓ Split: {len(corpus)} corpus rows, {len(queries)} query rows (disjoint)")

## 11. Stage → Parquet → Spark temp views (PyArrow, no PythonRDD)

Both subsets get a Parquet staging file + a Spark temp view. Bypassing
`spark.createDataFrame(list_of_rows)` avoids `PythonRDD` socket traffic, which
matters at scale (see parent [`README.md`](../README.md) "Why we stage
through Parquet").

In [ ]:
def stage_to_parquet(rows, embedding_dim, path):
    schema = pa.schema([
        pa.field("image_id", pa.string(), nullable=False),
        pa.field("category", pa.string(), nullable=False),
        pa.field("category_sanitized", pa.string(), nullable=False),
        pa.field("label", pa.int32(), nullable=False),
        pa.field("description", pa.string(), nullable=True),
        pa.field("image_bytes_raw", pa.binary(), nullable=False),
        pa.field("width", pa.int32(), nullable=False),
        pa.field("height", pa.int32(), nullable=False),
        pa.field("embedding",
                 pa.list_(pa.field("element", pa.float32(), nullable=False), list_size=embedding_dim),
                 nullable=False),
    ])
    cols = {f.name: [r[f.name] if f.name != "image_bytes_raw" else r["image_bytes_raw"] for r in rows]
            for f in schema}
    pq.write_table(pa.table(cols, schema=schema), path)

corpus_parquet = f"/tmp/staging_pets_batch_corpus_{BASE_FILE_FORMAT}.parquet"
queries_parquet = f"/tmp/staging_pets_batch_queries_{BASE_FILE_FORMAT}.parquet"

stage_to_parquet(corpus, embedding_dim, corpus_parquet)
spark.read.parquet(corpus_parquet).createOrReplaceTempView("staging_corpus")
print(f"✓ staging_corpus view registered ({corpus_parquet})")

stage_to_parquet(queries, embedding_dim, queries_parquet)
spark.read.parquet(queries_parquet).createOrReplaceTempView("staging_queries")
print(f"✓ staging_queries view registered ({queries_parquet})")

## 12. CREATE TABLE — corpus + queries

Identical DDL for both tables. The two share schema by design — this also
exercises the clashing-column rename path in
`HoodieVectorSearchPlanBuilder.buildBatchQueryPlan` (the query side's
`image_id` / `category` / `image_bytes` get the `_hudi_query_` prefix in
the TVF result).

In [ ]:
def create_table(name, path):
    spark.sql(f"DROP TABLE IF EXISTS {name}")
    spark.sql(f"""
        CREATE TABLE {name} (
            image_id            STRING,
            category            STRING,
            category_sanitized  STRING,
            label               INT,
            description         STRING,
            image_bytes         BLOB           COMMENT 'Pet image bytes (INLINE)',
            width               INT,
            height              INT,
            embedding           VECTOR({embedding_dim})
                                               COMMENT 'Image embedding for ANN search'
        ) USING hudi
        PARTITIONED BY (category_sanitized)
        LOCATION '{path}'
        TBLPROPERTIES (
            primaryKey = 'image_id',
            preCombineField = 'image_id',
            type = 'cow',
            'hoodie.table.base.file.format' = '{BASE_FILE_FORMAT}',
            'hoodie.write.record.merge.custom.implementation.classes' = 'org.apache.hudi.DefaultSparkRecordMerger'
        )
    """)
    print(f"✓ Created table {name} at {path}")

create_table(CONFIG["corpus_table_name"],  CONFIG["corpus_table_path"])
create_table(CONFIG["queries_table_name"], CONFIG["queries_table_path"])

## 13. INSERT INTO — corpus + queries

`named_struct('type', 'INLINE', 'data', <bytes>, 'reference', null)` is the
canonical shape for a Hudi BLOB INLINE value.

In [ ]:
def insert_from(view, table):
    spark.sql(f"""
        INSERT INTO {table}
        SELECT
            image_id, category, category_sanitized, label, description,
            named_struct(
                'type',      'INLINE',
                'data',      image_bytes_raw,
                'reference', cast(null as {BLOB_REFERENCE_CAST})
            ) AS image_bytes,
            width, height, embedding
        FROM {view}
    """)
    c = spark.sql(f"SELECT COUNT(image_id) AS c FROM {table}").collect()[0]["c"]
    print(f"✓ Inserted {c} rows into {table}")

insert_from("staging_corpus",  CONFIG["corpus_table_name"])
insert_from("staging_queries", CONFIG["queries_table_name"])

## 14. Batch search — `hudi_vector_search_batch` TVF

This is the line being certified. Both tables share column names, so every
non-embedding column from the query side gets the `_hudi_query_` prefix in
the output.

In [ ]:
sql = f"""
    SELECT
        image_id              AS corpus_image_id,
        category              AS corpus_category,
        image_bytes           AS corpus_image_bytes,
        _hudi_query_image_id  AS query_image_id,
        _hudi_query_category  AS query_category,
        _hudi_distance,
        _hudi_query_index
    FROM hudi_vector_search_batch(
        '{CONFIG["corpus_table_name"]}',
        'embedding',
        '{CONFIG["queries_table_name"]}',
        'embedding',
        {CONFIG["top_k"]},
        'cosine'
    )
    ORDER BY _hudi_query_index, _hudi_distance
"""
print(sql.strip())

tvf_rows = spark.sql(sql).collect()
print(f"\n✓ TVF returned {len(tvf_rows)} rows ({CONFIG['n_queries']} queries × {CONFIG['top_k']} matches)")

## 15. Schema sanity check

Confirm the prefix-rename code path produced the expected `_hudi_query_*`
columns. The renaming logic lives at
[`HoodieVectorSearchPlanBuilder.scala:305-310`](../../../../../../../hudi-spark-datasource/hudi-spark-common/src/main/scala/org/apache/spark/sql/hudi/analysis/HoodieVectorSearchPlanBuilder.scala).

In [ ]:
expected = {"corpus_image_id", "corpus_category", "corpus_image_bytes",
            "query_image_id", "query_category", "_hudi_distance", "_hudi_query_index"}
actual = set(tvf_rows[0].asDict().keys()) if tvf_rows else set()
assert expected.issubset(actual), f"missing {expected - actual}; got {actual}"
print(f"✓ TVF output columns: {sorted(actual)}")

## 16. Oracle validation — numpy ground truth

This is the load-bearing assertion of the notebook. We recompute cosine
distances from the same Python-side embeddings in numpy and assert the TVF
returns the same top-K per query (as a multiset, with per-rank distances
within `1e-5`). Position-wise comparison is multiset-based because tied
distances can be ordered differently by numpy and the JVM.

In [ ]:
corpus_embs = np.asarray([r["embedding"] for r in corpus], dtype=np.float32)
query_embs  = np.asarray([r["embedding"] for r in queries], dtype=np.float32)
corpus_ids  = np.asarray([r["image_id"]  for r in corpus])
query_ids   = np.asarray([r["image_id"]  for r in queries])

# Use float64 to match the JVM-side DenseVector(Double) arithmetic, then clamp
# to [0, 2] like VectorDistanceUtils.
sim = corpus_embs.astype(np.float64) @ query_embs.astype(np.float64).T
dist = np.clip(1.0 - sim, 0.0, 2.0)

tvf_by_query = {}
for r in tvf_rows:
    tvf_by_query.setdefault(r["query_image_id"], []).append(r)

assert set(tvf_by_query.keys()) == set(query_ids.tolist()), "query id sets disagree"

k = CONFIG["top_k"]
tol = CONFIG["oracle_distance_tol"]
max_delta = 0.0
matches = 0
for j, qid in enumerate(query_ids):
    order = np.argsort(dist[:, j], kind="stable")[:k]
    expected_ids   = corpus_ids[order].tolist()
    expected_dists = dist[order, j].tolist()

    tvf_for_q = tvf_by_query[qid]
    assert len(tvf_for_q) == k, f"query {qid}: got {len(tvf_for_q)} rows, expected {k}"
    tvf_ids   = [r["corpus_image_id"] for r in tvf_for_q]
    tvf_dists = [float(r["_hudi_distance"]) for r in tvf_for_q]

    assert sorted(tvf_ids) == sorted(expected_ids), (
        f"query {qid}:\n  expected ids: {expected_ids}\n  tvf ids     : {tvf_ids}")
    for i, (ed, td) in enumerate(zip(expected_dists, tvf_dists)):
        delta = abs(ed - td)
        max_delta = max(max_delta, delta)
        assert delta <= tol, f"query {qid} rank {i}: delta {delta:.2e} > tol {tol:.0e}"
    matches += 1

print(f"✓ ORACLE: {matches}/{len(query_ids)} queries match numpy ground truth")
print(f"  Max per-rank distance delta: {max_delta:.2e} (tol {tol:.0e})")

## 17. Render result panel

One row per query: query image on the left, then top-K matches with cosine
distance overlays. The numpy oracle (cell 16) is the load-bearing
correctness check — this panel is for eyeball sanity (same-breed / similar-breed).

In [ ]:
out_dir = Path(CONFIG["output_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

by_query = {}
for r in tvf_rows:
    by_query.setdefault(r["query_image_id"], []).append(r)

k = CONFIG["top_k"]
n = len(queries)
fig, axes = plt.subplots(n, k + 1, figsize=(2.0 * (k + 1), 2.2 * n))
if n == 1:
    axes = np.array([axes])

for row_idx, q in enumerate(queries):
    ax = axes[row_idx, 0]
    ax.imshow(Image.open(io.BytesIO(q["image_bytes_raw"])).convert("RGB"))
    ax.set_title(f"QUERY\n{q['category']}", fontsize=8, fontweight="bold")
    ax.axis("off")
    matches = by_query.get(q["image_id"], [])
    for col_idx in range(k):
        ax = axes[row_idx, col_idx + 1]
        if col_idx < len(matches):
            m = matches[col_idx]
            inline = m["corpus_image_bytes"]["data"] if m["corpus_image_bytes"] is not None else None
            if inline is not None:
                ax.imshow(Image.open(io.BytesIO(bytes(inline))).convert("RGB"))
            ax.set_title(f"{m['corpus_category']}\nd={float(m['_hudi_distance']):.3f}", fontsize=7)
        ax.axis("off")

plt.tight_layout()
panel_path = out_dir / CONFIG["panel_filename"]
plt.savefig(str(panel_path), dpi=120, bbox_inches="tight")
plt.show()
print(f"✓ Panel saved: {panel_path.resolve()}")

## 18. Summary

In [ ]:
print("=" * 80)
print(f"✓ Corpus:       {CONFIG['n_corpus']} rows @ {CONFIG['corpus_table_path']}")
print(f"✓ Queries:      {CONFIG['n_queries']} rows @ {CONFIG['queries_table_path']}")
print(f"✓ Embeddings:   {CONFIG['embedding_model']} (dim={embedding_dim})")
print(f"✓ Base format:  {CONFIG['base_file_format']}")
print(f"✓ TVF rows:     {len(tvf_rows)} ({CONFIG['n_queries']}×{CONFIG['top_k']})")
print(f"✓ Oracle delta: max {max_delta:.2e} (tol {tol:.0e})")
print(f"✓ Panel:        {panel_path.resolve()}")
print("=" * 80)
print("CERTIFIED ✓  hudi_vector_search_batch matches numpy ground truth")
print("=" * 80)

## 19. Stop Spark (optional)

In [ ]:
spark.stop()